[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# BSON Types


## What you will be able to do

Say which Python values MongoDB can store and which it refuses, and recognize the one refusal that
is not a refusal at all. Handle dates correctly: know that BSON keeps milliseconds and not
microseconds, that a client built without `tz_aware=True` hands you naive datetimes, and what that
costs you later. Store money without float rounding. And get documents out of MongoDB and into JSON,
which does not work by default and has two different fixes depending on whether anything has to read
it back.


## The idea

### The problem

A document looks like a `dict`, so it is natural to assume any `dict` is a document. It is not. BSON
has its own list of types, and a Python value that is not one of them either raises or is quietly
converted into something close enough, and "close enough" is the half that costs you.

### What BSON is

The binary format MongoDB stores and speaks. It covers what JSON covers, plus the things JSON
cannot: a 64 bit integer, a date, binary data, a decimal, and an `ObjectId`. It does not cover a
`set`, a `date` without a time, a `complex`, or Python's `Decimal`.

### Why a date loses precision

BSON stores a date as milliseconds since the epoch, in a 64 bit integer. A Python `datetime` holds
microseconds. Three digits have nowhere to go, so they are dropped on the way in, silently, and the
value you read back is not the value you wrote.

### Where this shows up

At the edges. Writing is where the conversions happen, and reading is where a naive datetime meets
an aware one and raises. Then again at the boundary with the outside world: `json.dumps` on a
document is the single most common first error in a web service built on MongoDB.

### What this notebook covers

The types that raise, and the one that does not. Dates: precision, `tz_aware`, and the comparison
that fails. `Decimal128` against `float`. Then `json.dumps`, `bson.json_util`, and the
`InvalidId` you get when a URL path segment reaches `ObjectId()`.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import datetime as dt

import pymongo

client = pymongo.MongoClient("mongodb://127.0.0.1:27017/shop", tz_aware=True)
shop = client.get_default_database()

shop.types.drop()
went_in = {
    "_id": "one",
    "when": dt.datetime(2026, 3, 4, 5, 6, 7, 891234, tzinfo=dt.timezone.utc),
    "pair": (1, 2),
}
shop.types.insert_one(dict(went_in))               # a copy, so went_in keeps its tuple
came_back = shop.types.find_one({"_id": "one"})

print("microseconds in: ", went_in["when"].microsecond)
print("microseconds out:", came_back["when"].microsecond)
print("a tuple went in and a", type(came_back["pair"]).__name__, "came out")
print("so what came back equals what went in:", came_back == went_in)
client.close()
```

```
microseconds in:  891234
microseconds out: 891000
a tuple went in and a list came out
so what came back equals what went in: False
```

Nothing raised, nothing warned, and the document that came back is not the document that went in.
Three digits of the timestamp are gone and the tuple is a list. Both are documented, both are
permanent, and neither is visible unless you compare.


## Setup

Twelve imports, MongoDB, and the boot cell.

- `bson` with `ObjectId` and `Decimal128`, which are the two BSON types with no Python equivalent
- `datetime` and `decimal` supply the values that do not survive the trip unchanged
- `json` is imported as `stdlib_json` so it cannot be confused with `bson.json_util`
- `subprocess`, `os`, `sys`, `time`, `random`, `version` and `PackageNotFoundError` run the boot cell

The boot cell is the same one **Why Documents** explains line by line.


In [1]:
import datetime as dt
import decimal
import json as stdlib_json
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import bson
import pymongo
from bson import Decimal128, ObjectId

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### What MongoDB will not store

Four common Python values that have no BSON equivalent:


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.types.drop()

for label, value in (("a set", {1, 2}),
                     ("a date", dt.date(2026, 1, 1)),
                     ("a Decimal", decimal.Decimal("1.5")),
                     ("a complex", 1 + 2j)):
    try:
        shop.types.insert_one({"v": value})
        print(f"  {label:11} stored")
    except bson.errors.InvalidDocument as error:
        print(f"  {label:11} -> {str(error).split(', of type')[0]}")


  a set       -> Invalid document: cannot encode object: {1, 2}
  a date      -> Invalid document: cannot encode object: datetime.date(2026, 1, 1)
  a Decimal   -> Invalid document: cannot encode object: Decimal('1.5')
  a complex   -> Invalid document: cannot encode object: (1+2j)


Each of those has an obvious replacement, and picking it is a decision you should make rather than
discover: a `set` becomes a sorted list, a `date` becomes a `datetime` at midnight, and a `Decimal`
becomes a `Decimal128`.

### And the one it stores anyway

This is the one to watch, because it does not raise:


In [3]:
shop.types.insert_one({"_id": "tuple", "pair": (1, 2), "nested": ({"a": 1}, {"b": 2})})
back = shop.types.find_one({"_id": "tuple"})

print("pair came back as: ", type(back["pair"]).__name__, back["pair"])
print("nested as well:    ", type(back["nested"]).__name__)
print("and it is not equal to what you wrote:", back["pair"] == (1, 2))


pair came back as:  list [1, 2]
nested as well:     list
and it is not equal to what you wrote: False


A tuple is a sequence, so PyMongo encodes it as a BSON array, and a BSON array decodes to a `list`.
Nothing is lost except the type, which matters exactly when your code does `isinstance(x, tuple)` or
uses the value as a dictionary key after reading it back.

### Dates, and the three digits that are gone

BSON keeps milliseconds:


In [4]:
exact = dt.datetime(2026, 3, 4, 5, 6, 7, 891234, tzinfo=dt.timezone.utc)
shop.types.insert_one({"_id": "when", "at": exact})
stored = shop.types.find_one({"_id": "when"})["at"]

print("wrote: ", exact.isoformat())
print("read:  ", stored.isoformat())
print("equal:", stored == exact, "| difference in microseconds:", exact.microsecond -
      stored.microsecond)


wrote:  2026-03-04T05:06:07.891234+00:00
read:   2026-03-04T05:06:07.891000+00:00
equal: False | difference in microseconds: 234


Two hundred and thirty four microseconds, gone. It is rarely important and it is occasionally very
important: a test that writes a timestamp and asserts it comes back equal fails, and the reason is
not in the test.

Truncate on your side if equality matters, so that the value you hold is the value stored:


In [5]:
def to_millis(when):
    """The same instant, rounded the way BSON will round it anyway."""
    return when.replace(microsecond=(when.microsecond // 1000) * 1000)


rounded = to_millis(exact)
shop.types.insert_one({"_id": "when2", "at": rounded})
print("now they agree:", shop.types.find_one({"_id": "when2"})["at"] == rounded)


now they agree: True


### tz_aware, and the comparison that fails

Every date in MongoDB is UTC. What differs is whether PyMongo tells you so:


In [6]:
aware = pymongo.MongoClient(URI, tz_aware=True)                     # what this guide always uses
naive = pymongo.MongoClient(URI)                                    # the default

with_tz = aware.get_default_database().types.find_one({"_id": "when"})["at"]
without = naive.get_default_database().types.find_one({"_id": "when"})["at"]

print("tz_aware=True: ", with_tz.isoformat(), "| tzinfo:", with_tz.tzinfo)
print("the default:   ", without.isoformat(), "| tzinfo:", without.tzinfo)


tz_aware=True:  2026-03-04T05:06:07.891000+00:00 | tzinfo: FixedOffset(datetime.timedelta(0), 'UTC')
the default:    2026-03-04T05:06:07.891000 | tzinfo: None


The same eight bytes on disk, read two ways. The naive one is not in local time and never was: it is
UTC with the label removed, which is worse than either, because nothing downstream can tell.

The bill arrives at a distance, usually in a comparison:


In [7]:
try:
    print(without > dt.datetime.now(dt.timezone.utc))
except TypeError as error:
    print("TypeError:", error)

print()
print("with tz_aware=True it just works:", with_tz < dt.datetime.now(dt.timezone.utc))
naive.close()


TypeError: can't compare offset-naive and offset-aware datetimes

with tz_aware=True it just works: True


Build the client with `tz_aware=True` once, at the top, and this whole class of failure disappears.
It is the reason every `MongoClient` in this guide has it.

### Money

A `float` cannot hold `0.1`, and three of them do not make `0.3`:


In [8]:
shop.types.insert_one({"_id": "money", "as_float": 0.1, "as_decimal": Decimal128("0.1")})
money = shop.types.find_one({"_id": "money"})

print("float:      ", money["as_float"] * 3)
print("Decimal128: ", money["as_decimal"].to_decimal() * 3)
print("what it is: ", type(money["as_decimal"]).__name__, "and it round trips exactly:",
      str(money["as_decimal"]) == "0.1")


float:       0.30000000000000004
Decimal128:  0.3
what it is:  Decimal128 and it round trips exactly: True


`Decimal128` is BSON's 128 bit decimal. It is not arithmetic: you call `to_decimal()` to get a
Python `Decimal`, do the arithmetic there, and wrap it in `Decimal128` again to store it.

It is worth the friction for money and worth nothing anywhere else. The seeded `price` field in this
guide is a plain float, which is the usual and usually fine choice.

### Out of MongoDB and into JSON

The most common first error in a service built on this database:


In [9]:
document = {"_id": ObjectId(), "kind": "laptop", "when": dt.datetime.now(dt.timezone.utc)}

try:
    stdlib_json.dumps(document)
except TypeError as error:
    print("TypeError:", error)

print()
print("and a seeded product is fine, because its _id is an int:",
      "kind" in stdlib_json.loads(stdlib_json.dumps(
          shop.products.find_one({"_id": 0}, {"_id": 1, "kind": 1}))))


TypeError: Object of type ObjectId is not JSON serializable

and a seeded product is fine, because its _id is an int: True


There are two fixes and they are not interchangeable.

For an API response, convert at the edge, because the caller wants a string and will never hand it
back as BSON:


In [10]:
def for_the_wire(document):
    """What a JSON API should send: plain types, ids as strings."""
    out = dict(document)
    if isinstance(out.get("_id"), ObjectId):
        out["_id"] = str(out["_id"])
    for field, value in out.items():
        if isinstance(value, dt.datetime):
            out[field] = value.isoformat()
    return out


sent = stdlib_json.dumps(for_the_wire(document))
print("keys sent:", sorted(stdlib_json.loads(sent)))
print("the id is now a", type(stdlib_json.loads(sent)["_id"]).__name__, "of",
      len(stdlib_json.loads(sent)["_id"]), "characters")


keys sent: ['_id', 'kind', 'when']
the id is now a str of 24 characters


For storing or moving BSON as text, where something must read it back as BSON, use
`bson.json_util`, which encodes the types rather than flattening them:


In [11]:
text = bson.json_util.dumps(document)
restored = bson.json_util.loads(text)

print("what the id looks like in that text:", sorted(stdlib_json.loads(text)["_id"]))
print("and it comes back as an", type(restored["_id"]).__name__)
print("identical to the original:", restored["_id"] == document["_id"])
print("the datetime too:", type(restored["when"]).__name__)


what the id looks like in that text: ['$oid']
and it comes back as an ObjectId
identical to the original: True
the datetime too: datetime


`{"$oid": ...}` is MongoDB's Extended JSON. It is the right answer for a dump file or a message
between two services that both speak BSON, and the wrong answer for a public API, where it leaks
your storage format to every caller.

### When to reach for which

| The Python value | What to store | Why |
|---|---|---|
| `set` | a sorted `list` | BSON has no set, and sorting makes it comparable |
| `datetime.date` | a `datetime` at midnight UTC | BSON has no date-only type |
| `decimal.Decimal` | `Decimal128` | exact, and the only exact option |
| money as `float` | `Decimal128`, if it must balance | `0.1 * 3` is not `0.3` |
| `tuple` | a `list`, deliberately | it becomes one anyway, silently |
| a document to a browser | `str(_id)` and `isoformat()` | the caller wants JSON, not BSON |
| a document to a file or another service | `bson.json_util.dumps` | it comes back as BSON |
| a datetime out of the database | a client with `tz_aware=True` | naive UTC is a bug waiting |

The default is `tz_aware=True` on every client, plain floats for quantities, `Decimal128` for money,
and conversion at the edge rather than `json_util` unless something really does read it back.

### One document, safely stored and safely sent, finished


In [12]:
def storable(record):
    """Turn a Python record into something BSON will take, with no surprises left."""
    out = {}
    for field, value in record.items():
        if isinstance(value, set):
            out[field] = sorted(value)
        elif isinstance(value, dt.datetime):
            out[field] = to_millis(value)
        elif isinstance(value, dt.date):                            # after datetime, which is a date
            out[field] = dt.datetime(value.year, value.month, value.day,
                                     tzinfo=dt.timezone.utc)
        elif isinstance(value, decimal.Decimal):
            out[field] = Decimal128(value)
        elif isinstance(value, tuple):
            out[field] = list(value)
        else:
            out[field] = value
    return out


record = {
    "_id": "order-1",
    "tags": {"gift", "priority"},
    "placed": dt.datetime(2026, 3, 4, 5, 6, 7, 891234, tzinfo=dt.timezone.utc),
    "due": dt.date(2026, 3, 11),
    "total": decimal.Decimal("129.90"),
    "sizes": (1, 2, 3),
}

shop.types.replace_one({"_id": "order-1"}, storable(record), upsert=True)
stored = shop.types.find_one({"_id": "order-1"})

for field in sorted(stored):
    print(f"  {field:7} {type(stored[field]).__name__:12} {stored[field]}")
print()
print("it round trips:", shop.types.find_one({"_id": "order-1"}) == stored)


  _id     str          order-1
  due     datetime     2026-03-11 00:00:00+00:00
  placed  datetime     2026-03-04 05:06:07.891000+00:00
  sizes   list         [1, 2, 3]
  tags    list         ['gift', 'priority']
  total   Decimal128   129.90

it round trips: True


Every conversion above is one you chose rather than one that happened to you, which is the whole
point. The `tuple` branch is there even though BSON would have taken it, because a rule with a hole
in it is the kind that surprises somebody later.

Note the order of the two date branches: a `datetime` **is** a `date` as far as `isinstance` is
concerned, so testing for `date` first would convert every timestamp to midnight.

### Where each part came from

| In `storable` | What it relies on | The section that showed it |
|---|---|---|
| `sorted(value)` for a set | BSON having no set type | What MongoDB will not store |
| `to_millis` | BSON keeping milliseconds | Dates, and the three digits that are gone |
| `datetime` at midnight for a date | BSON having no date-only type | What MongoDB will not store |
| `Decimal128(value)` | exact decimal arithmetic | Money |
| `list(value)` for a tuple | the conversion that happens silently | And the one it stores anyway |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/03-bson-types-solutions.ipynb).

**1.** Try to store a `set` and print the error it raises.


In [13]:
# your code here


**2.** Store a tuple and show what type comes back.


In [14]:
# your code here


**3.** Write a datetime with microseconds and show how many survive.


In [15]:
# your code here


**4.** Read the same date with and without `tz_aware=True`.


In [16]:
# your code here


**5.** Show that `Decimal128` adds up where `float` does not.


In [17]:
# your code here


**6.** Send a document with an `ObjectId` through `json.dumps`, both ways.


In [18]:
# your code here


## Common errors

### bson.errors.InvalidDocument: cannot encode object


In [19]:
shop.types.insert_one({"tags": {"gift"}})


InvalidDocument: Invalid document: cannot encode object: {'gift'}, of type: <class 'set'>

A `set` is the one people meet first, because deduplicating tags with a set is the obvious thing to
do in Python. The message names the value and its type, which is more than most encoders give you.
One tag is enough to refuse the whole document.

Sort it, and you get a list that is also comparable and indexable:


In [20]:
shop.types.replace_one({"_id": "tags"},
                       {"_id": "tags", "tags": sorted({"priority", "gift", "gift"})},
                       upsert=True)
print("stored:", shop.types.find_one({"_id": "tags"})["tags"])
print("and a query against an array still works:",
      shop.types.count_documents({"tags": "gift"}))


stored: ['gift', 'priority']
and a query against an array still works: 2


### TypeError: Object of type ObjectId is not JSON serializable


In [21]:
stdlib_json.dumps(shop.types.find_one({"_id": "tags"}) | {"oid": ObjectId()})


TypeError: Object of type ObjectId is not JSON serializable

This is the error that greets almost everybody the first time a MongoDB document reaches a web
framework, and it is worth being precise about the cause: `json.dumps` is Python's, it knows nothing
about BSON, and `ObjectId` is not one of the types it handles.

The `default=` hook is the smallest fix when you control the call:


In [22]:
def as_text(value):
    if isinstance(value, (ObjectId, Decimal128)):
        return str(value)
    if isinstance(value, dt.datetime):
        return value.isoformat()
    raise TypeError(f"no rule for {type(value).__name__}")


sent = stdlib_json.loads(stdlib_json.dumps({"oid": ObjectId(), "at": dt.datetime(2026, 1, 1)},
                                           default=as_text))
for field in sorted(sent):
    print(f"  {field:4} {type(sent[field]).__name__:4} {len(sent[field]):2} characters")
print("every value became a string, which is what the caller wanted")


  at   str  19 characters
  oid  str  24 characters
every value became a string, which is what the caller wanted


### bson.errors.InvalidId: it must be a 12-byte input or a 24-character hex string


In [23]:
ObjectId("not-an-id")


InvalidId: 'not-an-id' is not a valid ObjectId, it must be a 12-byte input or a 24-character hex string

This arrives from the other direction: a URL path segment, a form field or a query parameter comes
in as a string, and something calls `ObjectId()` on it. The string is whatever the caller typed.

An id from outside your program is input, so validate it rather than trusting it:


In [24]:
def as_object_id(text):
    """None rather than an exception, because a bad id from a caller is a 404, not a 500."""
    return ObjectId(text) if ObjectId.is_valid(text) else None


print("a real one:  ", as_object_id(str(ObjectId())) is not None)
print("a typo:      ", as_object_id("not-an-id"))
print("empty:       ", as_object_id(""))
print("the right length but not hex:", as_object_id("z" * 24))


a real one:   True
a typo:       None
empty:        None
the right length but not hex: None


### No error: the datetime that went in naive


In [25]:
local = dt.datetime(2026, 3, 4, 5, 6, 7)                            # no tzinfo at all
shop.types.replace_one({"_id": "naive"}, {"_id": "naive", "at": local}, upsert=True)

print("wrote:", local.isoformat(), "with tzinfo", local.tzinfo)
print("read: ", shop.types.find_one({"_id": "naive"})["at"].isoformat())
print("it was taken to be UTC, which it may very well not have been")


wrote: 2026-03-04T05:06:07 with tzinfo None
read:  2026-03-04T05:06:07+00:00
it was taken to be UTC, which it may very well not have been


PyMongo assumes a naive datetime is UTC. If the value came from `datetime.now()` on a machine in
another timezone, it is now wrong in the database by that offset, permanently, and nothing anywhere
will say so.

Make every datetime aware at the point it is created, not at the point it is stored:


In [26]:
correct = dt.datetime(2026, 3, 4, 5, 6, 7, tzinfo=dt.timezone.utc)
shop.types.replace_one({"_id": "aware"}, {"_id": "aware", "at": correct}, upsert=True)

print("stored and read back:", shop.types.find_one({"_id": "aware"})["at"] == correct)
print("and dt.datetime.now(dt.timezone.utc) is the habit worth having")


stored and read back: True
and dt.datetime.now(dt.timezone.utc) is the habit worth having


In [27]:
shop.types.drop()
client.close()
aware.close()
print("tidied up and closed")


tidied up and closed


## Recap

- BSON is not JSON and a document is not any `dict`. A `set`, a `datetime.date`, a
  `decimal.Decimal` and a `complex` all raise `bson.errors.InvalidDocument`.
- A `tuple` does not raise. It is stored as an array and comes back a `list`, which is the
  conversion to watch because nothing reports it.
- BSON stores dates to the millisecond. A `datetime` with microseconds comes back changed, which
  breaks equality in tests long before it breaks anything else.
- Every date in MongoDB is UTC. `tz_aware=True` makes PyMongo say so; without it you get naive
  datetimes that raise `TypeError` when compared with aware ones.
- A naive datetime written to MongoDB is assumed to be UTC, so one built from local time is stored
  wrong with no error at all.
- `Decimal128` is exact and `float` is not. Use it for money and nothing else.
- `json.dumps` cannot serialize an `ObjectId`. Convert at the edge for an API, and use
  `bson.json_util` only when something has to read it back as BSON.
- `ObjectId()` on a string from outside raises `InvalidId`. Guard it with `ObjectId.is_valid`.


## What is next

**find and find_one** is reading: one document against a cursor you can walk only once, the
projection that keeps a result small, and the counting that is exact against the counting that is
instant.


---

&#8592; **Previous:** [Collections and Documents](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/02-collections-and-documents.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [find and find_one](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/04-find-and-find-one.ipynb) &#8594;
